# El agente: ¿encontró lo mismo que ustedes?
**10 minutos · demostración con corridas pregrabadas (sin claves ni conexión a un modelo)**

El agente recibió **lo mismo que ustedes en el caso 3**: las tablas, el SQL de los dos reportes, la definición acordada y la diferencia.
- Herramienta única: SQL de **solo lectura** (sin acceso a archivos ni a red).
- **Presupuesto**: 25 consultas.
- Regla: cada hallazgo debe citar las consultas que lo demuestran.

Lo evaluamos contra la **lista fija de causas sembradas**: cuántas encontró, cuántas consultas gastó y cuántas afirmaciones hizo sin evidencia.

> Esto es un **experimento**. Si el agente falla, también es un resultado.

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (funciona en Colab y en local)
import importlib.util, os, sys, subprocess, pathlib
REPO_URL = "https://github.com/jona866/taller-conciliacion-de-datos-datostada2026.git"
if "google.colab" in sys.modules:
    destino = pathlib.Path("/content/taller-conciliacion-de-datos-datostada2026")
    if not destino.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(destino)], check=True)
    os.chdir(destino)
    if any(importlib.util.find_spec(mod) is None for mod in ("duckdb", "pandas", "pytz", "pytest")):
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    if not pathlib.Path("datos").exists():
        subprocess.run([sys.executable, "generar_datos.py"], check=True, stdout=subprocess.DEVNULL)
else:
    raiz = pathlib.Path.cwd()
    while not (raiz / "nnc.py").exists() and raiz != raiz.parent:
        raiz = raiz.parent
    if not (raiz / "nnc.py").exists():
        raise FileNotFoundError("Abre el cuaderno desde el repositorio del taller.")
    os.chdir(raiz)
if not pathlib.Path("datos/manifiesto.json").exists():
    raise FileNotFoundError("Faltan los datos. Ejecuta python generar_datos.py en la raíz del repositorio.")
sys.path.insert(0, os.getcwd())
import nnc
con = nnc.conectar("caso3")
print("Listo ✅  Tablas:", ", ".join(t for (t,) in con.execute("SHOW TABLES").fetchall()))

In [ ]:
import json, pathlib, pandas as pd
sys.path.insert(0, "agente")
import evaluar

corridas = sorted(pathlib.Path("agente/corridas").glob("*.json"))
reales = [c for c in corridas if json.loads(c.read_text(encoding="utf-8")).get("tipo") == "real"]
archivo = (reales or corridas)[-1]
corrida = json.loads(archivo.read_text(encoding="utf-8"))
print("Corrida:", archivo.name, "·", corrida["modelo"])
if corrida.get("nota"):
    print("Nota:", corrida["nota"])

### Qué hizo, consulta por consulta

In [ ]:
notas = {r["despues_de"]: r["texto"] for r in corrida.get("razonamiento", [])}
pd.set_option("display.max_colwidth", 120)
pd.DataFrame([{"id": q["id"], "intención": q.get("intencion", ""), "filas": q.get("filas"),
               "lo que concluyó": notas.get(i + 1, "")} for i, q in enumerate(corrida["consultas"])])

### Su informe

In [ ]:
pd.DataFrame(corrida['informe']['hallazgos'])

### Evaluación contra las causas sembradas

In [ ]:
evaluar.imprimir(evaluar.evaluar(corrida), corrida)

### Verificar, no creer
El agente afirma una causa sin evidencia. Pónganla a prueba con **una** consulta. Si la causa fuera la zona horaria de los tickets, algunos tickets caerían en un día distinto al de su pedido.

In [ ]:
nnc.q(con, """
    SELECT COUNT(*) AS tickets,
           COUNT(*) FILTER (WHERE CAST(t.fecha_hora AS DATE) <> CAST(p.fecha_pedido_local AS DATE)) AS en_otro_dia,
           COUNT(*) FILTER (WHERE t.recibido_en > TIMESTAMP '2026-11-03 09:00:00')          AS recibidos_despues_del_corte
    FROM caja_sucursal t JOIN pedidos p USING (pedido_id)
""")

**Para discutir (2 min en pareja):**
1. ¿Qué encontró más rápido que ustedes? ¿Qué se le escapó?
2. Su puente suma cero. ¿Eso lo hace correcto?
3. ¿Qué prueba automática habría detenido su conclusión?

**El patrón que sí funciona en la práctica:** el agente **propone** hipótesis; las consultas reproducibles y las pruebas **deciden**.